# Part 4: Circuits

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 4.1 Circuit Lab

### A circuit simulator in twenty lines

A circuit is a list of steps, and running it means multiplying the state by one matrix per step. `on_qubit` places a one-qubit gate in the full space (as in chapter 3.1), and `controlled` builds a gate that acts on the target only where every control is 1. The GHZ circuit from this chapter comes out as expected.

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
X = np.array([[0, 1], [1, 0]])


def on_qubit(gate, q, n):
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, gate if k == q else np.eye(2))
    return m


def controlled(gate, controls, target, n):
    """Apply gate to target in the part of the state where every control qubit is 1."""
    dim = 2 ** n
    full = on_qubit(gate, target, n)
    m = np.zeros((dim, dim), dtype=complex)
    for j in range(dim):                                   # column j: where |j> goes
        bits = [(j >> (n - 1 - q)) & 1 for q in range(n)]  # q0 is the leftmost bit
        m[:, j] = full[:, j] if all(bits[c] for c in controls) else np.eye(dim)[:, j]
    return m


def run(circuit, n):
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1
    for step in circuit:
        psi = step(n) @ psi
    return psi


ghz = [lambda n: on_qubit(H, 0, n),
       lambda n: controlled(X, [0], 1, n),
       lambda n: controlled(X, [1], 2, n)]
psi = run(ghz, 3)
for k, a in enumerate(psi):
    if abs(a) > 1e-12:
        print(f"|{k:03b}>  amplitude {a.real:+.4f}   probability {abs(a) ** 2:.2f}")

Only |000⟩ and |111⟩ survive, each with probability 1/2, exactly as in the Circuit Lab.

### Exercise 4.1.1: Build the W state

Use the same tools to build the W state (|001⟩ + |010⟩ + |100⟩)/√3, the "W state" exercise of the Circuit Lab. Recipe: Ry(θ) on q0 with cos(θ/2) = √(2/3); then H on q1, but only where q0 is 0; then X on q2, but only where q0 and q1 are both 0.

<details><summary>Hint</summary>

"Only where a qubit is 0" is a control on 1 sandwiched between two X gates on that qubit. Write a helper that wraps `controlled` in those X gates.

</details>

In [ ]:
# Your code here

### Exercise 4.1.2: The same circuits in Qiskit (optional)

If you have Qiskit installed (`pip install qiskit`), build the GHZ circuit there and print its probabilities. Then build a circuit with a single X on qubit 0 and look at the label Qiskit prints. Which end of the label is qubit 0?

<details><summary>Hint</summary>

`Statevector(qc).probabilities_dict()` returns a dictionary from labels to probabilities. The Circuit Lab's "Qiskit code" tab writes this kind of code for you.

</details>

In [ ]:
# Your code here

## 4.2 Interference

### Adding up paths, and why it equals matrix multiplication

For the circuit H, P(φ), H on |0⟩, the code adds up the amplitude of every path that ends at each result (Feynman's rule) and compares the answer with plain matrix multiplication.

In [ ]:
import itertools

import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)


def P(phi):
    return np.diag([1, np.exp(1j * phi)])


for phi in [0, np.pi / 2, np.pi]:
    gates = [H, P(phi), H]                     # in the order they are applied
    by_paths = np.zeros(2, dtype=complex)
    for path in itertools.product([0, 1], repeat=3):     # the state after each gate
        amp, prev = 1, 0                                  # every path starts in |0>
        for gate, now in zip(gates, path):
            amp *= gate[now, prev]                        # amplitude to go from prev to now
            prev = now
        by_paths[path[-1]] += amp
    by_matrices = H @ P(phi) @ H @ np.array([1, 0])
    print(f"phi = {phi:.4f}:  P(0) by paths = {abs(by_paths[0]) ** 2:.4f}, "
          f"by matrices = {abs(by_matrices[0]) ** 2:.4f}, cos^2(phi/2) = {np.cos(phi / 2) ** 2:.4f}")

The two methods agree, and P(0) = cos²(φ/2): certain at φ = 0, a coin flip at φ = π/2, and impossible at φ = π.

### Exercise 4.2.1: A marker that learns the path

Add a second qubit, the marker, that starts in |0⟩. Between H and P(φ), rotate the marker by Ry(χ), but only when the path qubit is |1⟩. For χ = 0, π/2 and π, find the largest and smallest P(0) as φ varies. Their difference is the fringe visibility V; compare it with cos(χ/2).

<details><summary>Hint</summary>

The controlled rotation is |0⟩⟨0| ⊗ I + |1⟩⟨1| ⊗ Ry(χ). P(0) for the path qubit adds the probabilities of |00⟩ and |01⟩.

</details>

In [ ]:
# Your code here

### Exercise 4.2.2: Looking at the path destroys the interference

Simulate H, then a measurement, then H, on |0⟩, for 2,000 runs. How often does the final result read 0? Compare with the same circuit without the middle measurement.

<details><summary>Hint</summary>

A measurement collapses the state to |0⟩ or |1⟩ with the Born-rule odds. Do that with `rng.choice`, then apply the second H to the collapsed state.

</details>

In [ ]:
# Your code here

## 4.3 Identities and universality

### Checking identities, up to a global phase

Two circuits are equivalent when their matrices agree up to a global phase. `equivalent(A, B)` divides out the phase and compares; the checks below are identities from this chapter.

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Z = np.diag([1, -1])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
S = np.diag([1, 1j])
P0, P1 = np.diag([1, 0]), np.diag([0, 1])
CNOT_01 = np.kron(P0, I) + np.kron(P1, X)      # control q0, target q1
CNOT_10 = np.kron(I, P0) + np.kron(X, P1)      # control q1, target q0
SWAP = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]])
CZ = np.diag([1, 1, 1, -1])


def rz(t):
    return np.diag([np.exp(-1j * t / 2), np.exp(1j * t / 2)])


def equivalent(A, B):
    k = np.argmax(np.abs(B))                     # a nonzero entry of B
    phase = A.flat[k] / B.flat[k]
    return bool(np.isclose(abs(phase), 1) and np.allclose(A, phase * B))


HH = np.kron(H, H)
checks = {
    "HXH = Z": (H @ X @ H, Z),
    "three CNOTs = SWAP": (CNOT_01 @ CNOT_10 @ CNOT_01, SWAP),
    "H on both flips a CNOT": (HH @ CNOT_01 @ HH, CNOT_10),
    "H CNOT H on the target = CZ": (np.kron(I, H) @ CNOT_01 @ np.kron(I, H), CZ),
    "Rz(pi/2) = S, up to phase": (rz(np.pi / 2), S),
    "S = Z (wrong on purpose)": (S, Z),
}
for name, (A, B) in checks.items():
    print(f"{name:30s} {equivalent(A, B)}")

### Exercise 4.3.1: What H and S can reach, and what H and T can reach

Starting from |0⟩, apply every sequence of up to 12 gates chosen from {H, S}, and count the different states you reach (ignoring global phase). Then do the same with {H, T}. Why does the first count stop growing?

<details><summary>Hint</summary>

Represent a state by its Bloch vector rounded to 6 decimals, which ignores the global phase, and collect them in a Python set. Grow the set one gate at a time instead of listing every sequence.

</details>

In [ ]:
# Your code here

### Exercise 4.3.2: The Toffoli gate from CNOTs and T gates

Build the standard decomposition of the Toffoli gate (2 H, 6 CNOT and 7 T or T† gates, the "Toffoli from CNOT + T" example in the Circuit Lab) as an 8 × 8 matrix, and check that it equals the Toffoli matrix exactly.

<details><summary>Hint</summary>

Write a helper that turns a one-qubit gate on qubit q, or a CNOT from c to t, into an 8 × 8 matrix, then multiply the matrices in circuit order: each new gate goes on the left.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 4.1.1: Build the W state

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
X = np.array([[0, 1], [1, 0]])


def on_qubit(gate, q, n):
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, gate if k == q else np.eye(2))
    return m


def controlled(gate, controls, target, n):
    dim = 2 ** n
    full = on_qubit(gate, target, n)
    m = np.zeros((dim, dim), dtype=complex)
    for j in range(dim):
        bits = [(j >> (n - 1 - q)) & 1 for q in range(n)]
        m[:, j] = full[:, j] if all(bits[c] for c in controls) else np.eye(dim)[:, j]
    return m


def controlled_on_zero(gate, controls, target, n):
    flips = np.eye(2 ** n)
    for c in controls:
        flips = on_qubit(X, c, n) @ flips
    return flips @ controlled(gate, controls, target, n) @ flips


def ry(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]])


n = 3
psi = np.zeros(8, dtype=complex)
psi[0] = 1
psi = on_qubit(ry(2 * np.arccos(np.sqrt(2 / 3))), 0, n) @ psi
psi = controlled_on_zero(H, [0], 1, n) @ psi
psi = controlled_on_zero(X, [0, 1], 2, n) @ psi
for k, a in enumerate(psi):
    if abs(a) > 1e-12:
        print(f"|{k:03b}>  {a.real:+.4f}")
print("each amplitude should be 1/sqrt(3) =", round(1 / np.sqrt(3), 4))

Expected output:

```
|001>  +0.5774
|010>  +0.5774
|100>  +0.5774
each amplitude should be 1/sqrt(3) = 0.5774
```

### Solution 4.1.2: The same circuits in Qiskit (optional)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector



def probabilities(qc):
    """Qiskit's probabilities as a plain dictionary {label: probability}."""
    return {str(k): round(float(v), 4) for k, v in Statevector(qc).probabilities_dict().items()}


ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
print("GHZ:         ", probabilities(ghz))

flip = QuantumCircuit(3)
flip.x(0)
print("X on qubit 0:", probabilities(flip))

Expected output:

```
GHZ:          {'000': 0.5, '111': 0.5}
X on qubit 0: {'001': 1.0}
```

Qiskit writes qubit 0 as the rightmost character, so X on qubit 0 gives the label '001', which this course writes as |100⟩. Keep that in mind when you compare results.

### Solution 4.2.1: A marker that learns the path

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
I = np.eye(2)


def ry(t):
    return np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])


def P(phi):
    return np.diag([1, np.exp(1j * phi)])


def p0(phi, chi):
    marker = np.kron(np.diag([1, 0]), I) + np.kron(np.diag([0, 1]), ry(chi))
    psi = np.kron(H @ [1, 0], [1, 0])                    # path in |+>, marker in |0>
    psi = np.kron(H, I) @ np.kron(P(phi), I) @ marker @ psi
    return abs(psi[0]) ** 2 + abs(psi[1]) ** 2           # path reads 0, any marker


for chi in [0, np.pi / 2, np.pi]:
    values = [p0(phi, chi) for phi in np.linspace(0, 2 * np.pi, 361)]
    visibility = (max(values) - min(values))
    print(f"chi = {chi:.4f}:  P(0) from {min(values):.3f} to {max(values):.3f}, "
          f"visibility {visibility:.3f}, cos(chi/2) = {np.cos(chi / 2) + 0:.3f}")

Expected output:

```
chi = 0.0000:  P(0) from 0.000 to 1.000, visibility 1.000, cos(chi/2) = 1.000
chi = 1.5708:  P(0) from 0.146 to 0.854, visibility 0.707, cos(chi/2) = 0.707
chi = 3.1416:  P(0) from 0.500 to 0.500, visibility 0.000, cos(chi/2) = 0.000
```

The more the marker learns (larger χ), the smaller the fringes. At χ = π the marker records the path perfectly and the interference is gone: P(0) = 1/2 for every φ.

### Solution 4.2.2: Looking at the path destroys the interference

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=8)
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
basis = [np.array([1, 0]), np.array([0, 1])]


def measure(psi):
    p = np.abs(psi) ** 2
    return rng.choice([0, 1], p=p / p.sum())


zeros_measured = 0
for _ in range(2000):
    middle = measure(H @ basis[0])               # the measurement picks a path
    zeros_measured += measure(H @ basis[middle]) == 0
p0_unmeasured = abs((H @ H @ basis[0])[0]) ** 2
print(f"with the middle measurement:    P(0) is about {zeros_measured / 2000:.3f}")
print(f"without the middle measurement: P(0) = {p0_unmeasured:.3f}")

Expected output:

```
with the middle measurement:    P(0) is about 0.501
without the middle measurement: P(0) = 1.000
```

### Solution 4.3.1: What H and S can reach, and what H and T can reach

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
S = np.diag([1, 1j])
T = np.diag([1, np.exp(1j * np.pi / 4)])


def key(psi):
    r = [np.vdot(psi, P @ psi).real for P in (X, Y, Z)]
    return tuple(np.round(r, 6) + 0)


for name, gates in [("H and S", [H, S]), ("H and T", [H, T])]:
    frontier = {key(np.array([1, 0])): np.array([1, 0], dtype=complex)}
    seen = dict(frontier)
    counts = []
    for length in range(1, 13):
        new = {}
        for psi in frontier.values():
            for g in gates:
                out = g @ psi
                k = key(out)
                if k not in seen:
                    new[k] = out
        seen.update(new)
        frontier = new
        counts.append(len(seen))
    print(f"{name}: states reached with up to 1, 2, ..., 12 gates: {counts}")

Expected output:

```
H and S: states reached with up to 1, 2, ..., 12 gates: [2, 3, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6]
H and T: states reached with up to 1, 2, ..., 12 gates: [2, 3, 5, 8, 13, 19, 26, 35, 48, 64, 85, 113]
```

H and S reach only the six landmark states, because every product of H and S belongs to a finite group, the single-qubit Clifford group (24 rotations of the sphere, ignoring global phase). With T the count keeps growing, and the states spread over the whole sphere: that is why T makes {H, T, CNOT} universal.

### Solution 4.3.2: The Toffoli gate from CNOTs and T gates

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
T = np.diag([1, np.exp(1j * np.pi / 4)])
Tdg = T.conj().T


def one(gate, q, n=3):
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, gate if k == q else I)
    return m


def cnot(c, t, n=3):
    return one(np.diag([1, 0]), c, n) + one(np.diag([0, 1]), c, n) @ one(X, t, n)


circuit = [one(H, 2), cnot(1, 2), one(Tdg, 2), cnot(0, 2), one(T, 2), cnot(1, 2), one(Tdg, 2), cnot(0, 2),
           one(T, 1), one(T, 2), one(H, 2), cnot(0, 1), one(T, 0), one(Tdg, 1), cnot(0, 1)]
U = np.eye(8)
for gate in circuit:
    U = gate @ U                                   # later gates multiply from the left

toffoli = np.eye(8)
toffoli[[6, 7]] = toffoli[[7, 6]]                  # swap |110> and |111>
print("equals the Toffoli matrix:", np.allclose(U, toffoli))
print("gates used:", len(circuit))

Expected output:

```
equals the Toffoli matrix: True
gates used: 15
```